In [1]:
!pip install google-genai pydantic pandas

In [5]:
from typing import Literal
from pydantic import BaseModel, Field
import pandas as pd
from google import genai
from google.genai import types
import json

# 1. CONNECT TO THE AI
# Replace the text inside quotes with your actual API key
API_KEY = "Your_API_KEY"
client = genai.Client(api_key=API_KEY)

# 2. DEFINE THE STRICT STRUCTURE (SCHEMA)
class ReviewAnalysis(BaseModel):
    category: Literal["Delivery Issue", "Food Quality", "Payment/Billing", "App Bug", "Positive Experience", "Other"] = Field(
        description="The main category of the customer's feedback"
    )
    sentiment: Literal["Positive", "Neutral", "Negative"] = Field(
        description="The overall emotional tone of the review"
    )
    urgency_level: Literal["Low", "Medium", "High"] = Field(
        description="High for payment failures or spoiled food, Medium for delays, Low for compliments"
    )
    key_issue_summary: str = Field(
        description="A crisp 1-sentence summary of the user's issue"
    )
    suggested_action: str = Field(
        description="What the operations or support team should do to fix this"
    )

# 3. SAMPLE CUSTOMER REVIEWS
customer_reviews = [
    "I ordered biryani 90 minutes ago. It still hasn't arrived and the delivery partner is not answering calls!",
    "My money got debited twice for order #49281, but the app shows order failed. Please refund my 650 rupees immediately.",
    "The packaging was neat, and the pasta was hot and fresh. Loved the extra cheese!",
    "The app crashed three times when I tried to apply the flat 50% discount coupon at checkout."
]

# 4. PROCESS REVIEWS THROUGH GEMINI
# `analyzed_results` and `index` are available from the previous execution
print(f"Processing customer reviews from index {index - 1} through AI...\n")

for i, review in enumerate(customer_reviews[index - 1:], start=index):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Analyze this customer feedback:\n\"{review}\"",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ReviewAnalysis,
            temperature=0.0  # Zero randomness for strict, deterministic output
        ),
    )

    parsed_data = json.loads(response.text)
    parsed_data["Original_Review"] = review
    analyzed_results.append(parsed_data)
    print(f"✓ Processed Review {i}/{len(customer_reviews)}")

# 5. DISPLAY AS A CLEAN TABLE & SAVE TO CSV
df = pd.DataFrame(analyzed_results)
columns_order = ["Original_Review", "category", "sentiment", "urgency_level", "key_issue_summary", "suggested_action"]
df = df[columns_order]

print("\n--- FINAL STRUCTURED DATA ---")
display(df)

df.to_csv("customer_feedback_analysis.csv", index=False)
print("\nFile saved successfully as 'customer_feedback_analysis.csv'!")

Processing customer reviews from index 2 through AI...

✓ Processed Review 3/4
✓ Processed Review 4/4

--- FINAL STRUCTURED DATA ---


,Original_Review,category,sentiment,urgency_level,key_issue_summary,suggested_action
0,I ordered biryani 90 minutes ago. It still has...,Delivery Issue,Negative,High,Customer's biryani order is 90 minutes late an...,"Contact the delivery partner immediately, prov..."
1,"My money got debited twice for order #49281, b...",Payment/Billing,Negative,High,Customer was double-debited 650 rupees for ord...,Investigate the double debit for order #49281 ...
2,"The packaging was neat, and the pasta was hot ...",Positive Experience,Positive,Low,Customer was highly satisfied with the food qu...,Acknowledge positive feedback and consider sha...
3,The app crashed three times when I tried to ap...,App Bug,Negative,Medium,The app crashed multiple times when the user a...,Investigate the app's stability during checkou...



File saved successfully as 'customer_feedback_analysis.csv'!
